# CS383: Data Science and Machine Learning
## Lecture 10 Exercises — Decision Trees, Random Forests, and Gradient Boosting

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 10 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

Run this first — same row-level dataset from the lecture: `is_critical` is the target, `score` /
`boro` / `cuisine_top` are the features.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "score": rng.integers(0, 71, size=n),
    })
    inspections_df["is_critical"] = rng.integers(0, 2, size=n)
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_top", "score", "is_critical"]].head()

---

### Step 1 — Split first, then check the baseline

Same order as always: split before you touch anything else. Then determine the majority class from the
*training* labels only, and evaluate it against the test labels -- that's the number any real model has
to beat.

In [ ]:
X = inspections_df[["score", "boro", "cuisine_top"]]
y = inspections_df["is_critical"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=__________, random_state=383, stratify=y)

majority_class = y_train.mode()[0]
majority_baseline = (y_test == majority_class).mean()
print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

### Step 2 — Build a `ColumnTransformer` and fit a decision tree

One-hot encode the two categorical columns -- no scaler needed this time, since trees don't care about
feature scale. Fit a `DecisionTreeClassifier` with `max_depth=4`, same as the lecture.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("cat", __________(handle_unknown="ignore"), ["boro", "cuisine_top"]),
], remainder="passthrough")

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=383)
tree_model.fit(X_train_ready, y_train)

print("Training features shape:", X_train_ready.shape)

### Step 3 — Evaluate the decision tree

In [ ]:
tree_pred = tree_model.predict(X_test_ready)
tree_accuracy = __________(y_test, tree_pred)

print(f"Decision tree accuracy: {tree_accuracy:.3f}  (baseline: {majority_baseline:.3f})")

### Step 4 — Fit a random forest at a few different sizes

Try `n_estimators` = 50, 200, and 500, keeping `max_depth=4` fixed each time. Watch how much (or how
little) accuracy actually changes as you add more trees.

In [ ]:
for n_est in [50, 200, 500]:
    forest_model = RandomForestClassifier(n_estimators=__________, max_depth=4, random_state=383)
    forest_model.fit(X_train_ready, y_train)
    forest_pred = forest_model.predict(X_test_ready)
    forest_accuracy = accuracy_score(y_test, forest_pred)
    print(f"n_estimators={n_est:>3}: accuracy = {forest_accuracy:.3f}  (baseline: {majority_baseline:.3f}, tree: {tree_accuracy:.3f})")

### Step 5 — Feature importance from the random forest

Use the `n_estimators=500` forest (re-fit it, or reuse it if your loop above left it in memory) and plot
its top features.

In [ ]:
forest_model = RandomForestClassifier(n_estimators=500, max_depth=4, random_state=383)
forest_model.fit(X_train_ready, y_train)

feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(forest_model.__________, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
importances.head(10).plot(kind="barh", color="#2B6CB0")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Random Forest Feature Importances (top 10)")
plt.tight_layout()
plt.show()

importances.head(5)

### Step 6 — Explain it back

In 2-3 sentences: how much did accuracy change between `n_estimators=50` and `n_estimators=500`? Was it
worth the extra computation? And which feature came out on top in the importance ranking -- does that
match what you'd expect from the lecture?

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why does a decision tree's decision boundary always come out boxy -- made of straight, right-angle
   edges -- no matter what the data looks like?
2. What goes wrong if you fit a `DecisionTreeClassifier` with no `max_depth` limit at all? What's the
   giveaway sign that it happened?
3. In your own words, what's the difference between how a random forest builds its trees (bagging) and
   how gradient boosting builds its trees? Why does that difference matter?
4. A feature has the highest importance score in your random forest. Name one reason that feature might
   *not* actually be the most important thing in the real world.
5. What question do you still have about tree-based models heading into Lecture 11 (Pipelines &
   Tuning)?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same workflow to a different real target: NYC 311's `resolution_time_hours`, turned into a
yes/no question -- was a complaint resolved **within 24 hours**? (The same target Lecture 8's Optional
Challenge used.)

### Setup — NYC 311

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
complaints_df = complaints_df[complaints_df["resolution_time_hours"] >= 0].reset_index(drop=True)

# The target: did this complaint get resolved within 24 hours?
complaints_df["fast_resolution"] = (complaints_df["resolution_time_hours"] <= 24).astype(int)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
print(f"fast_resolution balance: {complaints_df['fast_resolution'].mean():.3f}")

### Step 1 — Split, encode, and fit

Use `complaint_type`, `borough`, and `hour_filed` as features and `fast_resolution` as the target. Split
(don't forget `stratify=`), determine the majority class from `y_train` only, build a `ColumnTransformer`
(one-hot the two categoricals, pass `hour_filed` through), then fit a `DecisionTreeClassifier` and a
`RandomForestClassifier` (`max_depth=6` for both).

In [ ]:
# Your code here



### Step 2 — Compare

How does this compare to the restaurant dataset in the main Lab? Which dataset's tree models cleared
their baseline by a wider margin?

In [ ]:
# Your code here



### Step 3 — Reflect

Both this dataset and the restaurant dataset ask a real yes/no question about real NYC data -- but one
almost certainly classified far better than the other, for both the linear model back in Lecture 8 *and*
the tree models here. Which one, and why do you think that is? (Hint: think about what `complaint_type`
alone might be telling you here that `score` alone couldn't tell you about `is_critical`.)

**Your answer:**

### Big idea
> A more expressive model -- one that can carve curved or boxy boundaries instead of a single straight
> line -- is not automatically a better one. It can extract more of the real signal that's actually
> sitting in the features (as you likely just saw with 311), but it can just as easily spend its extra
> flexibility chasing noise instead, especially when the real signal is thin (as with the restaurant
> data). Always ask whether a model's extra complexity bought you a better fit to the *pattern*, or just
> a better fit to this one particular training set.